# Run Pi 0.5 on LIBERO

**On first session, run:** Sections 1 → 2 → 3 (full install, ~15 min) for setup, 4 for verification, 5 for benchmarking

**Every subsequent session:** Sections 1 → 2 -> 3b (restore from cache, shorter than Section 3) → 4 → 5 → 6

**Note:** Drive caching may not work depending on how much drive storage is available (~12GB)

On an **A100**, each forward pass seeems to take 20-60 seconds (time is highly variable)

---
## Section 1: GPU check + Drive mount

Run every session.

In [1]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

NVIDIA A100-SXM4-40GB, 40960 MiB
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [2]:
!pip show huggingface_hub | grep Version
!grep -n "dataclass\|@strict" /usr/local/lib/python3.12/dist-packages/lerobot/policies/groot/groot_n1.py | head

Version: 1.16.1
grep: /usr/local/lib/python3.12/dist-packages/lerobot/policies/groot/groot_n1.py: No such file or directory


In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE      = '/content/drive/MyDrive'

# NOTE: I know this is called smolvla_colab_cache, just ignore it for now, it was from the smolvla notebook
# and the packages carry over

CACHE_DIR  = f'{DRIVE}/smolvla_colab_cache'   # pip + package snapshot lives here
HF_HOME    = f'{DRIVE}/smolvla_colab_cache/hf_models'  # HuggingFace models
RESULTS_DIR= f'{DRIVE}/smolvla_results'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Point HF at Drive so model downloads persist across sessions
os.environ['HF_HOME'] = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'

SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
snapshot_exists = os.path.exists(SNAPSHOT)
print(f'Cache dir:  {CACHE_DIR}')
print(f'HF models:  {HF_HOME}')
print(f'Snapshot:   {"FOUND — use Section 3b (fast restore)" if snapshot_exists else "not found — run Section 3 (full install) first"}')

Mounted at /content/drive
Cache dir:  /content/drive/MyDrive/smolvla_colab_cache
HF models:  /content/drive/MyDrive/smolvla_colab_cache/hf_models
Snapshot:   not found — run Section 3 (full install) first


---
## Section 2: System libraries

Run **once per session** (only ~2 min — apt-get is fast and can't be cached to Drive easily).

In [4]:
%%bash
apt-get update -qq
apt-get install -y -qq libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev libegl1-mesa-dev patchelf ffmpeg
# EGL fix for Colab
mkdir -p /usr/share/glvnd/egl_vendor.d
echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}' \
    > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
echo 'System deps done'

System deps done


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


---
## Section 3: Full install (FIRST SESSION ONLY)

Run this only on your very first session, or if the snapshot becomes stale.
At the end it saves a snapshot to Drive. **Skip to Section 3b on subsequent sessions.**

In [5]:
!pip install \
    mujoco robosuite libero sentencepiece tiktoken \
    "transformers==5.5.4" \
    "lerobot[pi0] @ git+https://github.com/huggingface/lerobot.git@01dcb4c29222bc9f2388cebf87f0e79965a9508b"

  Cloning https://github.com/huggingface/lerobot.git (to revision 01dcb4c29222bc9f2388cebf87f0e79965a9508b) to /tmp/pip-install-cxad9sv6/lerobot_a2509b6f8bbb47aa9c45629ced790598
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/lerobot.git /tmp/pip-install-cxad9sv6/lerobot_a2509b6f8bbb47aa9c45629ced790598
  Running command git rev-parse -q --verify 'sha^01dcb4c29222bc9f2388cebf87f0e79965a9508b'
  Running command git fetch -q https://github.com/huggingface/lerobot.git 01dcb4c29222bc9f2388cebf87f0e79965a9508b
  Running command git checkout -q 01dcb4c29222bc9f2388cebf87f0e79965a9508b
  Resolved https://github.com/huggingface/lerobot.git to commit 01dcb4c29222bc9f2388cebf87f0e79965a9508b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB

In [6]:
# Verify that packages have installed properly (easy for conflicts to occur)

import torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
print("PI05 import: OK")

from libero.libero import benchmark
print("LIBERO import: OK")

import mujoco
print(f"MuJoCo: {mujoco.__version__}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


PI05 import: OK
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /usr/local/lib/python3.12/dist-packages/libero/libero
bddl_files: /usr/local/lib/python3.12/dist-packages/libero/libero/./bddl_files
init_states: /usr/local/lib/python3.12/dist-packages/libero/libero/./init_files
datasets: /usr/local/lib/python3.12/dist-packages/libero/libero/../datasets
assets: /usr/local/lib/python3.12/dist-packages/libero/libero/./assets
LIBERO import: OK
MuJoCo: 3.9.0
Device: cuda


In [7]:
# Login to HuggingFace (needed for gated PaliGemma tokenizer used by pi0.5)
from huggingface_hub import login, snapshot_download
import os

# Paste your HuggingFace token when prompted
# Get one at: https://huggingface.co/settings/tokens
# Also accept PaliGemma license at: https://huggingface.co/google/paligemma-3b-pt-224
login()

print(f'Downloading pi0.5 LIBERO checkpoint to {os.environ["HF_HOME"]} ...')
path = snapshot_download(repo_id='lerobot/pi05_libero_finetuned')
print(f'Checkpoint cached at: {path}')

# Pre-download PaliGemma tokenizer
print('Downloading PaliGemma tokenizer...')
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224')
print('PaliGemma tokenizer cached.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 7473.10 MB. The target location /content/drive/MyDrive/smolvla_colab_cache/hf_models/hub/models--lerobot--pi05_libero_finetuned/blobs only has 1291.24 MB free disk space.
  warnings.warn(


Checkpoint cached at: /content/drive/MyDrive/smolvla_colab_cache/hf_models/hub/models--lerobot--pi05_libero_finetuned/snapshots/dbf8a3f794a9c4297b44f40b752712f50073d945
PaliGemma tokenizer cached.


In [8]:
# Save snapshot of installed packages to Drive
# This is the step that makes future sessions fast (~2-3 min restore vs 15-20 min reinstall)
import subprocess, os, sys, shutil

print('Saving package snapshot locally first, then moving to Drive...')
LOCAL_SNAPSHOT = '/content/site_packages.tar.gz'
DRIVE_SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'

# Save site-packages automatically detecting python version
site_packages_dir = f'usr/local/lib/python3.{sys.version_info.minor}/dist-packages'

# 1. Tar to local disk (very fast)
result = subprocess.run([
    'tar', '-czf', LOCAL_SNAPSHOT,
    '-C', '/',
    site_packages_dir,
    'usr/local/bin',       # includes lerobot-train, lerobot-eval etc.
], capture_output=True, text=True)

if result.returncode == 0:
    # 2. Copy to Drive
    print('Copying to Google Drive...')
    shutil.copy(LOCAL_SNAPSHOT, DRIVE_SNAPSHOT)
    os.remove(LOCAL_SNAPSHOT) # clean up local copy

    size_mb = os.path.getsize(DRIVE_SNAPSHOT) / 1e6
    print(f'Snapshot saved: {DRIVE_SNAPSHOT} ({size_mb:.0f} MB)')
    print('Future sessions can use Section 3b (fast restore)')
else:
    print('Warning: snapshot failed. Check error:')
    print(result.stderr[:500])

Saving package snapshot locally first, then moving to Drive...
Copying to Google Drive...
Snapshot saved: /content/drive/MyDrive/smolvla_colab_cache/site_packages.tar.gz (11041 MB)
Future sessions can use Section 3b (fast restore)


---
## Section 3b: Fast restore (SUBSEQUENT SESSIONS)

Run this instead of Section 3 on every session after the first.
Restores packages from Drive snapshot in ~6 minutes.

In [12]:
import subprocess, sys, os, time, shutil, importlib

SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
LOCAL_SNAPSHOT = '/content/site_packages_restore.tar.gz'

if not os.path.exists(SNAPSHOT):
    raise FileNotFoundError(
        'No snapshot found. Run Section 3 (full install) first to create it.'
    )

t0 = time.time()
size_mb = os.path.getsize(SNAPSHOT) / 1e6
print(f'Found snapshot on Drive: {size_mb:.0f} MB')

print('Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...')
shutil.copy(SNAPSHOT, LOCAL_SNAPSHOT)

print('Extracting packages...')
result = subprocess.run(
    ['tar', '-xzf', LOCAL_SNAPSHOT, '-C', '/'],
    capture_output=True, text=True
)
os.remove(LOCAL_SNAPSHOT) # Cleanup

if result.returncode != 0:
    print('Restore failed:', result.stderr[:500])
else:
    elapsed = time.time() - t0
    print(f'Restored in {elapsed:.1f}s')

# Force Python to recognize the newly extracted modules
importlib.invalidate_caches()

# Quick check
for pkg in ['mujoco', 'robosuite', 'libero', 'lerobot']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg} OK')
    except ImportError as e:
        print(f'  {pkg} MISSING: {e}')
print('Restore complete.')

Found snapshot on Drive: 11025 MB
Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...
Extracting packages...


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


Restored in 276.0s
  mujoco OK
  robosuite OK
  libero OK
  lerobot OK
Restore complete.


---
## Section 4: Verify

Quick sanity check before running evaluation -- honestly can just skip this in place of the video verification (see cells at end of Section 4)

In [10]:
# ── Configuration ─────────────────────────────
SUITE      = 'libero_spatial'  # libero_spatial | libero_object | libero_goal | libero_long
N_EPISODES = 10                 # per task. Use 1 for smoke test, 10 for proper eval
START_SEED = 0
MAX_STEPS  = 600
VERBOSE    = True
# ───────────────────────────────────────────────
print(f'{SUITE} | {N_EPISODES} ep/task | {N_EPISODES*10} total episodes')

libero_spatial | 10 ep/task | 100 total episodes


In [11]:
from transformers import AutoTokenizer
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Load model & tokenizer ──────────────────────────────────────────────
policy = PI05Policy.from_pretrained('lerobot/pi05_libero_finetuned')
policy = policy.to(device)
policy.eval()

tokenizer = AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224')
print(f'PI05 loaded. Params: {sum(p.numel() for p in policy.parameters())/1e6:.0f}M')


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi
Loading model from: lerobot/pi05_libero_finetuned


✓ Loaded state dict from model.safetensors
All keys loaded successfully!
PI05 loaded. Params: 4143M


In [12]:
import time, json, torch, numpy as np, os, math
from tqdm.notebook import tqdm
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from lerobot.policies.pi05.modeling_pi05 import PI05Policy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CAMERAS = ['agentview', 'robot0_eye_in_hand']
CAMERA_KEY_MAP = {
    'agentview':          'observation.images.image',
    'robot0_eye_in_hand': 'observation.images.image2',
}
IMG_SIZE = 360           # render resolution; policy resizes internally
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
NUM_STEPS_WAIT = 10

# pi0.5 expects an empty (zero) third camera
EMPTY_CAM_SHAPE = (1, 3, 224, 224)


def _quat2axisangle(quat):
    if quat[3] > 1.0:  quat[3] = 1.0
    elif quat[3] < -1.0: quat[3] = -1.0
    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den


from lerobot.policies.factory import make_pre_post_processors

# Create official pre/post processors after loading policy
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    'lerobot/pi05_libero_finetuned',
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

def obs_to_policy(obs_dict, task_desc, device):
    agentview = np.ascontiguousarray(obs_dict['agentview_image'][::-1, ::-1])
    wrist     = np.ascontiguousarray(obs_dict['robot0_eye_in_hand_image'][::-1, ::-1])

    img_agent = torch.from_numpy(agentview / 255.0).permute(2,0,1).float()
    img_wrist = torch.from_numpy(wrist     / 255.0).permute(2,0,1).float()

    state = np.concatenate([
        obs_dict['robot0_eef_pos'],
        _quat2axisangle(obs_dict['robot0_eef_quat']),
        obs_dict['robot0_gripper_qpos'],
    ])

    # Pass raw observation — preprocessor handles normalization, tokenization etc.
    return {
        'observation.images.image':  img_agent,
        'observation.images.image2': img_wrist,
        'observation.state': torch.from_numpy(state).float(),
        'task': task_desc,
    }

def run_episode(env, init_state, policy, task_desc, max_steps, device):
    env.reset()
    policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    t0 = time.time()
    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch = preprocess(raw_obs)        # official preprocessing
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)       # official postprocessing
        if isinstance(action, torch.Tensor):
            action = action.squeeze(0).cpu().numpy()
        obs, _, done, _ = env.step(action)
        if env.check_success():
            return True, step+1, time.time()-t0
        if done:
            break
    return False, step+1, time.time()-t0


policy = PI05Policy.from_pretrained('lerobot/pi05_libero_finetuned').to(device).eval()
task_suite = benchmark.get_benchmark_dict()[SUITE]()
print(f'Ready. {task_suite.n_tasks} tasks on {device}')


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading model from: lerobot/pi05_libero_finetuned


✓ Loaded state dict from model.safetensors
All keys loaded successfully!
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Ready. 10 tasks on cuda


Purely a debugging cell used before for SmolVLA, feel free to ignore

Verify that policy can consistently succeed on various tasks in LIBERO Spatial

In [14]:
all_results = []
suite_start = time.time()

# Task-specific max steps from reference script
MAX_STEPS_MAP = {
    'libero_spatial': 220,
    'libero_object':  280,
    'libero_goal':    300,
    'libero_10':      520,
    'libero_90':      400,
}
max_steps = MAX_STEPS_MAP.get(SUITE, 300)

for task_idx in range(task_suite.n_tasks):
    task = task_suite.get_task(task_idx)
    init_states = task_suite.get_task_init_states(task_idx)

    bddl_file_path = os.path.join(
        get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

    env = OffScreenRenderEnv(
        bddl_file_name=bddl_file_path, camera_names=CAMERAS,
        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
        has_offscreen_renderer=True, use_camera_obs=True,
        has_renderer=False, reward_shaping=False,
    )

    print(f'\nTask {task_idx+1}/{task_suite.n_tasks}: {task.language[:65]}...')
    n_success = 0

    for ep_idx in tqdm(range(min(N_EPISODES, len(init_states))), desc=f'  T{task_idx+1}'):
        success, n_steps, elapsed = run_episode(
            env, init_states[ep_idx], policy, task.language, max_steps, device)
        n_success += int(success)
        all_results.append({
            'suite': SUITE, 'task_idx': task_idx,
            'task_description': task.language, 'episode': ep_idx,
            'success': success, 'n_steps': n_steps, 'elapsed_s': round(elapsed, 2)
        })
        if VERBOSE:
            print(f'    {"✓" if success else "✗"} ep{ep_idx} | {n_steps} steps | {elapsed:.1f}s')

    env.close()
    print(f'  SR: {n_success/N_EPISODES:.0%} ({n_success}/{N_EPISODES})')

    ts = time.strftime('%Y%m%d_%H%M%S')
    with open(f'{RESULTS_DIR}/partial_{SUITE}_{ts}.json', 'w') as f:
        json.dump(all_results, f)

overall_sr = sum(r['success'] for r in all_results) / len(all_results)
total_min = (time.time() - suite_start) / 60
print(f'\nOverall SR: {overall_sr:.1%} | Time: {total_min:.1f} min')

Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1855: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


Assets downloaded successfully to /root/.cache/libero/assets

Task 1/10: pick up the black bowl between the plate and the ramekin and plac...


...

    ✓ ep0 | 126 steps | 7.0s
    ✓ ep1 | 126 steps | 6.8s
    ✓ ep2 | 123 steps | 6.7s
    ✓ ep3 | 116 steps | 6.3s
    ✓ ep4 | 115 steps | 6.1s
    ✓ ep5 | 125 steps | 6.9s
    ✓ ep6 | 127 steps | 6.7s
    ✓ ep7 | 120 steps | 6.4s
    ✓ ep8 | 126 steps | 7.1s
    ✓ ep9 | 119 steps | 6.6s
  SR: 100% (10/10)

Task 9/10: pick up the black bowl next to the plate and place it on the plat...


  T9:   0%|          | 0/10 [00:00<?, ?it/s]

    ✓ ep0 | 107 steps | 5.5s
    ✗ ep1 | 220 steps | 10.8s
    ✗ ep2 | 220 steps | 11.0s
    ✓ ep3 | 96 steps | 4.8s
    ✗ ep4 | 220 steps | 10.3s
    ✓ ep5 | 96 steps | 4.7s
    ✓ ep6 | 101 steps | 5.2s
    ✓ ep7 | 99 steps | 5.1s
    ✓ ep8 | 97 steps | 4.7s
    ✓ ep9 | 95 steps | 4.6s
  SR: 70% (7/10)

Task 10/10: pick up the black bowl on the wooden cabinet and place it on the ...


  T10:   0%|          | 0/10 [00:00<?, ?it/s]

    ✓ ep0 | 123 steps | 6.3s
    ✓ ep1 | 114 steps | 5.9s
    ✓ ep2 | 126 steps | 6.5s
    ✓ ep3 | 115 steps | 5.9s
    ✓ ep4 | 111 steps | 5.7s
    ✓ ep5 | 115 steps | 5.9s
    ✓ ep6 | 123 steps | 6.4s
    ✓ ep7 | 123 steps | 6.2s
    ✓ ep8 | 116 steps | 5.9s
    ✓ ep9 | 126 steps | 6.5s
  SR: 100% (10/10)

Overall SR: 83.0% | Time: 23.9 min


In [15]:
import imageio
from IPython.display import Video, display

task = task_suite.get_task(0)
bddl_file_path = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
init_states = task_suite.get_task_init_states(0)

video_env = OffScreenRenderEnv(
    bddl_file_name=bddl_file_path, camera_names=CAMERAS,
    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
    has_offscreen_renderer=True, use_camera_obs=True,
    has_renderer=False, reward_shaping=False,
)

try:
    video_env.reset()
    policy.reset()
    obs = video_env.set_init_state(init_states[0])
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = video_env.step(LIBERO_DUMMY_ACTION)

    frames = []
    print(f"Recording: '{task.language}'...")

    for step in range(220):
        frames.append(np.ascontiguousarray(obs['agentview_image'][::-1, ::-1]))

        raw_obs = obs_to_policy(obs, task.language, device)
        batch = preprocess(raw_obs)              # official preprocessing
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)             # official postprocessing
        if isinstance(action, torch.Tensor):
            action = action.squeeze(0).cpu().numpy()

        obs, _, done, _ = video_env.step(action)
        if video_env.check_success():
            print("Task succeeded!")
            break

    video_path = '/content/rollout.mp4'
    imageio.mimsave(video_path, frames, fps=30)
    display(Video(video_path, embed=True, width=512))
finally:
    video_env.close()

Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1855: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


Assets downloaded successfully to /root/.cache/libero/assets


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Recording: 'pick up the black bowl between the plate and the ramekin and place it on the plate'...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/torch/_inductor/select_algorithm.py:3686: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.s

Task succeeded!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## Section 4c: Self-refining (Predict-and-Perturb) sampling

Implements *Self-Refining Video Sampling* (arXiv:2601.18577) for Pi0.5's flow-matching
action sampler. The pretrained velocity field is treated as a denoising autoencoder, and at
chosen (high-noise) denoising steps we run an inner **Predict-and-Perturb (P&P)** loop:

- **predict** the clean action  `â = x − s·v(x, s)`
- **perturb** back to the same noise level  `x' ← (1−s)·â + s·ε`,  `ε∼N(0,I)`
- repeat `K` times.

**Convention:** LeRobot integrates time from `s=1.0` (pure noise) → `s=0.0` (clean action) — the
*opposite* of the paper's `t=0`-is-noise convention. So the paper's "early, high-noise" steps
(`t<0.2`) map to the **first** Euler steps here (large `s`): select them with `step_indices=(1,)` /
`(1,2)` or `time_min=0.8`. (Step 0 has `s=1.0`, where the perturb drops `â` and returns fresh
noise: *refinement* there is just a reseed, but *uncertainty* there is still meaningful — it's the
spread of the one-shot prediction over independent noise draws. Default starts at step 1 so the
refinement demo is non-trivial; use step 0 freely for `mode="uncertainty"`.)

**Scope (current):** uncertainty *measurement* + *unconditional* refinement (no mask / no
gating). The disagreement between consecutive clean predictions (Eq. 10) is always recorded for
analysis; refinement is applied only when `mode ∈ {refine, both}`. A mask / uncertainty-triggered
behaviour can be slotted in later without changing the structure.

We monkey-patch `policy.model.sample_actions` because P&P must run *inside* the Euler loop, which
`select_action` / `predict_action_chunk` don't expose. The original is saved as
`policy.model._orig_sample_actions`, and an equivalence test confirms the reimplemented loop
matches it when P&P is off.

In [33]:
# ── Predict-and-Perturb (P&P) sampler: config, recorder, patched Euler loop ──
import torch, numpy as np
from dataclasses import dataclass
from typing import Optional, Sequence
from lerobot.policies.pi05.modeling_pi05 import make_att_2d_masks


@dataclass
class PnPConfig:
    """Self-refining (Predict-and-Perturb) sampling config for Pi0.5 flow matching.

    LeRobot time runs s=1.0 (noise) -> s=0.0 (clean), so the paper's early/high-noise
    steps are the FIRST Euler steps (large s). Select them with step_indices=(1,)/(1,2)
    or time_min=0.8.

    NOTE on step 0 (s=1.0): the perturb (1-s)*a_hat + s*eps drops a_hat entirely and returns
    fresh noise. So REFINEMENT at s=1.0 is a no-op-like reseed (the map x->eps has no
    contraction). But UNCERTAINTY at s=1.0 is meaningful: each iteration predicts a_hat from
    an independent noise draw, so U measures the spread of the policy's one-shot action
    prediction over the noise prior (overall predictive variance given the observation).
    Default selection starts at step 1 so the *refinement* demo is non-trivial; step 0 is fine
    and informative for mode="uncertainty".
    """
    enabled: bool = False
    step_indices: Optional[Sequence[int]] = (1,)   # which Euler steps run P&P (ignored if time_min set)
    time_min: Optional[float] = None               # alt selector: run P&P when s >= time_min (note: excludes nothing below 1.0; s=1.0 only at step 0)
    num_iterations: int = 3                         # K predict-and-perturb iterations
    mode: str = "both"                              # "uncertainty" | "refine" | "both"
    action_dim: int = 7                             # real (un-padded) action dims used for uncertainty
    record_per_iteration: bool = False              # also store the full (K,B,chunk,adim) a_hat stack per step

    def step_selected(self, step: int, s: float) -> bool:
        if not self.enabled:
            return False
        if self.time_min is not None:
            return s >= self.time_min
        return self.step_indices is not None and step in tuple(self.step_indices)

    @property
    def do_refine(self) -> bool:
        return self.mode in ("refine", "both")


class PnPRecorder:
    """Collects per-episode P&P uncertainty so it can later be correlated with outcomes.

    After a run, `episodes` is a list of dicts:
        {"meta": {...}, "success": bool, "n_steps": int,
         "chunks": [ {"num_steps": int,
                      "steps": [ {"step": i, "s": float,
                                  "u_consecutive": np[B,chunk,adim],  # Eq.10 mean|Δâ|
                                  "a_std": np[B,chunk,adim],          # spread of â over iters
                                  "u_mean", "u_max", "a_std_mean": float,
                                  # only if cfg.record_per_iteration:
                                  "a_hats": np[K,B,chunk,adim]}, ... ]}, ... ]}
    One "chunk" == one full action-chunk prediction (one sample_actions call).
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.episodes = []
        self._cur = None

    def new_episode(self, meta=None):
        self._cur = {"meta": dict(meta or {}), "chunks": [], "success": None, "n_steps": None}

    def log_chunk(self, chunk_rec):
        if self._cur is not None:
            self._cur["chunks"].append(chunk_rec)

    def close_episode(self, success, n_steps):
        if self._cur is None:
            return
        self._cur["success"] = bool(success)
        self._cur["n_steps"] = int(n_steps)
        self.episodes.append(self._cur)
        self._cur = None


# Global handles (notebook-style); the patched method reads these each call.
PNP_CONFIG = PnPConfig()
PNP_RECORDER = PnPRecorder()


def _pnp_refine_at_step(x_t, s, vfield, cfg):
    """Run K predict-and-perturb iterations at fixed noise level s.

        predict:  a_hat = x - s * v(x, s)
        perturb:  x'    = (1 - s) * a_hat + s * eps,   eps ~ N(0, I)

    Returns (x_out, rec). x_out is the refined re-noised state if cfg.do_refine, else
    the original x_t unchanged (uncertainty-only is non-invasive). `rec` always holds the
    uncertainty measured across iterations (a free by-product of the predicts).
    """
    adim = cfg.action_dim
    # At s=1.0 the perturb drops a_hat (refinement = reseed), but uncertainty across the
    # independent draws is still meaningful. At 0<s<1 a_hat is partially retained.
    x_acc = x_t
    a_hats = []
    for _ in range(cfg.num_iterations):
        v = vfield(x_acc)
        a_hat = x_acc - s * v                       # predicted clean action
        a_hats.append(a_hat[..., :adim])
        eps = torch.randn_like(x_acc)
        x_acc = (1.0 - s) * a_hat + s * eps         # perturb back to level s (ends on perturb)

    A = torch.stack(a_hats, dim=0)                  # (K, B, chunk, adim)
    if A.shape[0] >= 2:
        u_consecutive = (A[1:] - A[:-1]).abs().mean(dim=0)   # (B, chunk, adim)  -- Eq.10 analog
        a_std = A.std(dim=0)                                 # (B, chunk, adim)
    else:
        u_consecutive = torch.zeros_like(A[0])
        a_std = torch.zeros_like(A[0])

    rec = {
        "s": float(s),
        "u_consecutive": u_consecutive.detach().float().cpu().numpy(),
        "a_std": a_std.detach().float().cpu().numpy(),
        "u_mean": float(u_consecutive.mean()),
        "u_max": float(u_consecutive.max()),
        "a_std_mean": float(a_std.mean()),
    }
    if cfg.record_per_iteration:
        # Full clean-prediction stack; derive any metric later (consecutive diffs, all-pairs,
        # convergence curves, ...). Shape (K, B, chunk, adim).
        rec["a_hats"] = A.detach().float().cpu().numpy()
    return (x_acc if cfg.do_refine else x_t), rec


@torch.no_grad()
def _sample_actions_pnp(self, images, img_masks, tokens, masks, noise=None, num_steps=None, **kwargs):
    """Drop-in replacement for PI05Pytorch.sample_actions with optional P&P refinement.

    Delegates to the saved original when P&P is disabled or under RTC; otherwise replicates
    the Euler loop verbatim and injects the P&P inner loop at the selected steps.
    """
    cfg = PNP_CONFIG
    if (not cfg.enabled) or self._rtc_enabled():
        return self._orig_sample_actions(
            images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs)

    if num_steps is None:
        num_steps = self.config.num_inference_steps
    bsize = tokens.shape[0]
    device = tokens.device
    if noise is None:
        actions_shape = (bsize, self.config.chunk_size, self.config.max_action_dim)
        noise = self.sample_noise(actions_shape, device)

    # ---- prefix / KV cache: replicated verbatim from the original sample_actions ----
    prefix_embs, prefix_pad_masks, prefix_att_masks = self.embed_prefix(images, img_masks, tokens, masks)
    prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    prefix_att_2d_masks_4d = self._prepare_attention_masks_4d(prefix_att_2d_masks)
    self.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = "eager"
    _, past_key_values = self.paligemma_with_expert.forward(
        attention_mask=prefix_att_2d_masks_4d,
        position_ids=prefix_position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

    dt = -1.0 / num_steps
    x_t = noise
    chunk_rec = {"num_steps": num_steps, "steps": []}

    for step in range(num_steps):
        time = 1.0 + step * dt
        s = time
        time_tensor = torch.tensor(time, dtype=torch.float32, device=device).expand(bsize)

        def vfield(input_x_t, _ts=time_tensor):
            return self.denoise_step(
                prefix_pad_masks=prefix_pad_masks,
                past_key_values=past_key_values,
                x_t=input_x_t,
                timestep=_ts,
            )

        if cfg.step_selected(step, s):
            x_t, rec = _pnp_refine_at_step(x_t, s, vfield, cfg)
            rec["step"] = step
            chunk_rec["steps"].append(rec)

        # normal Euler step (identical to original when no P&P ran this step)
        v_t = vfield(x_t)
        x_t = x_t + dt * v_t

    PNP_RECORDER.log_chunk(chunk_rec)
    return x_t

print("PnP defined: PnPConfig, PnPRecorder, PNP_CONFIG, PNP_RECORDER, _sample_actions_pnp")

PnP defined: PnPConfig, PnPRecorder, PNP_CONFIG, PNP_RECORDER, _sample_actions_pnp


In [34]:
# ── Apply the monkey-patch onto policy.model ──────────────────────────────
import types

# Infer real (un-padded) action dim for the uncertainty maps (LIBERO = 7).
try:
    from lerobot.constants import ACTION
    PNP_CONFIG.action_dim = policy.config.output_features[ACTION].shape[0]
except Exception as e:
    PNP_CONFIG.action_dim = 7
    print(f"(could not infer action_dim, defaulting to 7: {e})")

print(f"action_dim = {PNP_CONFIG.action_dim} | num_inference_steps = {policy.config.num_inference_steps}")

# Save original once, then patch. Idempotent across re-runs of this cell.
if not hasattr(policy.model, "_orig_sample_actions"):
    policy.model._orig_sample_actions = policy.model.sample_actions
policy.model.sample_actions = types.MethodType(_sample_actions_pnp, policy.model)

# The original sample_actions was compiled with torch.compile in PI05Pytorch.__init__.
# We must recompile the patched version with the same settings so that the reimplemented
# Euler loop runs in the same compiled context; without this, eager vs compiled numerical
# differences cause ~2-3% divergence in the prefix KV forward pass.
if getattr(policy.model.config, 'compile_model', False):
    policy.model.sample_actions = torch.compile(
        policy.model.sample_actions,
        mode=policy.model.config.compile_mode,
    )
    print(f"Recompiled patched sample_actions (mode={policy.model.config.compile_mode!r}).")

# Default to OFF so the rest of the notebook behaves exactly as before until you opt in.
PNP_CONFIG.enabled = False
print("Patched policy.model.sample_actions (PnP currently disabled).")

(could not infer action_dim, defaulting to 7: No module named 'lerobot.constants')
action_dim = 7 | num_inference_steps = 10
Recompiled patched sample_actions (mode='max-autotune').
Patched policy.model.sample_actions (PnP currently disabled).


In [35]:
torch.use_deterministic_algorithms(True)
import os; os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ── Equivalence test: reimplemented loop == original when P&P is off ───────
# (A) PnP disabled -> delegates to the saved original loop.
# (B) PnP enabled but NO step selected -> exercises OUR reimplemented loop; must match (A)
#     bit-for-bit given the same fixed noise. (C) sanity: refinement actually changes output.
task = task_suite.get_task(0)
bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
init_states = task_suite.get_task_init_states(0)

_env = OffScreenRenderEnv(
    bddl_file_name=bddl, camera_names=CAMERAS,
    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
    has_offscreen_renderer=True, use_camera_obs=True,
    has_renderer=False, reward_shaping=False,
)
try:
    _env.reset(); policy.reset()
    obs = _env.set_init_state(init_states[0])
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = _env.step(LIBERO_DUMMY_ACTION)
    batch = preprocess(obs_to_policy(obs, task.language, device))

    fixed_noise = policy.model.sample_noise(
        (1, policy.config.chunk_size, policy.config.max_action_dim), device)

    # (A) original loop (delegation)
    PNP_CONFIG.enabled = False
    with torch.no_grad():
        a_orig = policy.predict_action_chunk(batch, noise=fixed_noise.clone())


    # (B) our loop, P&P enabled but no step selected
    PNP_CONFIG.enabled = True
    PNP_CONFIG.step_indices = ()
    PNP_CONFIG.time_min = None
    with torch.no_grad():
        a_loop = policy.predict_action_chunk(batch, noise=fixed_noise.clone())

    max_abs = (a_orig - a_loop).abs().max().item()
    print(f"[loop-equivalence] max|Δ| = {max_abs:.3e}  ->", "OK" if max_abs < 1e-4 else "MISMATCH")
    assert torch.allclose(a_orig, a_loop, atol=1e-4), "Reimplemented loop diverged from original!"

    # (B2) P&P enabled WITH steps selected, but in measure-only mode -> non-invasive.
    #      The inner loop runs (and records uncertainty) yet must NOT change x_t, so the
    #      final action must still equal the original.
    PNP_CONFIG.step_indices = (1, 2)
    PNP_CONFIG.mode = "uncertainty"
    PNP_CONFIG.num_iterations = 3
    torch.manual_seed(0)
    with torch.no_grad():
        a_meas = policy.predict_action_chunk(batch, noise=fixed_noise.clone())
    max_abs2 = (a_orig - a_meas).abs().max().item()
    print(f"[non-invasive]     max|Δ| = {max_abs2:.3e}  ->", "OK" if max_abs2 < 1e-4 else "MISMATCH")
    assert torch.allclose(a_orig, a_meas, atol=1e-4), "uncertainty-only mode changed the output!"

    # (C) refinement-on sanity: output should actually move
    PNP_CONFIG.step_indices = (1,)
    PNP_CONFIG.mode = "refine"
    PNP_CONFIG.num_iterations = 3
    torch.manual_seed(0)
    with torch.no_grad():
        a_ref = policy.predict_action_chunk(batch, noise=fixed_noise.clone())
    print(f"[refine sanity]   max|Δ vs orig| = {(a_ref - a_orig).abs().max().item():.3e}  (expected > 0)")
finally:
    _env.close()
    PNP_CONFIG.enabled = False   # leave OFF by default
    PNP_CONFIG.step_indices = (1,)
    PNP_CONFIG.mode = "both"
print("Equivalence test passed; PnP left disabled.")

[loop-equivalence] max|Δ| = 0.000e+00  -> OK


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[non-invasive]     max|Δ| = 0.000e+00  -> OK


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

[refine sanity]   max|Δ vs orig| = 7.947e-01  (expected > 0)
Equivalence test passed; PnP left disabled.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [36]:
# ── Recording-aware rollout + small demo (uncertainty + outcomes) ──────────
def run_episode_pnp(env, init_state, policy, task_desc, max_steps, device,
                    ep_meta=None, db=None,
                    suite=None, task_idx=None, episode_idx=None):
    """Same as run_episode, but brackets the episode with PnP recorder hooks so every
    action-chunk's P&P uncertainty is tied to this episode's success/length.

    If db (RolloutDB) is provided, logs the episode automatically after completion.
    suite/task_idx/episode_idx are used to build the deterministic rollout_id.
    """
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    PNP_RECORDER.new_episode(ep_meta)
    t0 = time.time(); success = False; step = 0
    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch = preprocess(raw_obs)
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)
        if isinstance(action, torch.Tensor):
            action = action.squeeze(0).cpu().numpy()
        obs, _, done, _ = env.step(action)
        if env.check_success():
            success = True
            break
        if done:
            break
    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1)

    if db is not None:
        rollout_id = RolloutDB.make_rollout_id(
            suite or '', task_idx or 0, episode_idx or 0, init_state, PNP_CONFIG)
        db.log_episode(
            rollout_id=rollout_id,
            suite=suite or '',
            task_idx=task_idx or 0,
            task_desc=task_desc,
            episode_idx=episode_idx or 0,
            init_state=init_state,
            success=success,
            n_steps=step + 1,
            elapsed_s=elapsed,
            pnp_cfg=PNP_CONFIG,
            episode_rec=PNP_RECORDER.episodes[-1],
        )

    return success, step + 1, elapsed



In [37]:
# ── RolloutDB: SQLite-backed persistent store for rollouts + P&P uncertainty ─
import sqlite3, hashlib, json as _json, time as _time
import numpy as np

class RolloutDB:
    """SQLite store for rollout outcomes and per-step P&P uncertainty.

    Tables
    ------
    rollouts        — one row per episode (outcome, config, episode-level U stats)
    pnp_euler_steps — one row per (chunk × euler_step) with fine-grained U metrics

    Usage
    -----
    db = RolloutDB('/path/to/results.db')
    # pass db= to run_episode_pnp; it logs automatically
    succ, nstep, el = run_episode_pnp(..., db=db, suite=SUITE, task_idx=0, episode_idx=ep)
    db.summary()   # per-task success rate + mean uncertainty
    db.query('SELECT * FROM rollouts WHERE success=0')
    """

    _DDL = """
    CREATE TABLE IF NOT EXISTS rollouts (
        rollout_id        TEXT PRIMARY KEY,
        suite             TEXT,
        task_idx          INTEGER,
        task_desc         TEXT,
        episode_idx       INTEGER,
        init_state_hash   TEXT,
        success           INTEGER,
        n_steps           INTEGER,
        elapsed_s         REAL,
        pnp_enabled       INTEGER,
        pnp_k             INTEGER,
        pnp_step_indices  TEXT,
        pnp_mode          TEXT,
        u_mean_episode    REAL,
        u_max_episode     REAL,
        n_pnp_activations INTEGER,
        timestamp         TEXT
    );
    CREATE TABLE IF NOT EXISTS pnp_euler_steps (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        rollout_id   TEXT    NOT NULL REFERENCES rollouts(rollout_id),
        chunk_idx    INTEGER NOT NULL,
        euler_step   INTEGER NOT NULL,
        s            REAL,
        u_mean       REAL,
        u_max        REAL,
        a_std_mean   REAL
    );
    CREATE INDEX IF NOT EXISTS idx_pes_rollout ON pnp_euler_steps(rollout_id);
    """

    def __init__(self, db_path):
        self.db_path = str(db_path)
        self._con = sqlite3.connect(self.db_path, check_same_thread=False)
        self._con.executescript(self._DDL)
        self._con.commit()
        n = self._con.execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
        print(f"RolloutDB: {self.db_path}  ({n} existing rollouts)")

    # ── ID helpers ────────────────────────────────────────────────────────

    @staticmethod
    def init_state_hash(init_state):
        return hashlib.md5(np.asarray(init_state).tobytes()).hexdigest()[:12]

    @staticmethod
    def make_rollout_id(suite, task_idx, episode_idx, init_state, pnp_cfg):
        """Deterministic 16-char hex ID for a (task, episode, pnp_config) triple."""
        cfg_str = _json.dumps({
            'enabled':      pnp_cfg.enabled,
            'k':            pnp_cfg.num_iterations,
            'step_indices': list(pnp_cfg.step_indices) if pnp_cfg.step_indices else None,
            'time_min':     pnp_cfg.time_min,
            'mode':         pnp_cfg.mode,
        }, sort_keys=True)
        key = f'{suite}:{task_idx}:{episode_idx}:{RolloutDB.init_state_hash(init_state)}:{cfg_str}'
        return hashlib.sha256(key.encode()).hexdigest()[:16]

    # ── Write ─────────────────────────────────────────────────────────────

    def log_episode(self, rollout_id, suite, task_idx, task_desc, episode_idx,
                    init_state, success, n_steps, elapsed_s, pnp_cfg, episode_rec):
        """Write one episode + its per-step P&P records atomically."""
        all_step_recs = [
            (ci, st)
            for ci, chunk in enumerate(episode_rec.get('chunks', []))
            for st in chunk.get('steps', [])
        ]
        u_vals = [st['u_mean'] for _, st in all_step_recs]
        u_mean_ep = float(np.mean(u_vals)) if u_vals else None
        u_max_ep  = float(np.max(u_vals))  if u_vals else None

        if pnp_cfg.step_indices is not None:
            step_idx_str = _json.dumps(list(pnp_cfg.step_indices))
        else:
            step_idx_str = f'time_min:{pnp_cfg.time_min}'

        with self._con:
            self._con.execute(
                'INSERT OR REPLACE INTO rollouts VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
                (rollout_id, suite, task_idx, task_desc, episode_idx,
                 self.init_state_hash(init_state),
                 int(success), n_steps, round(elapsed_s, 3),
                 int(pnp_cfg.enabled), pnp_cfg.num_iterations,
                 step_idx_str, pnp_cfg.mode,
                 u_mean_ep, u_max_ep, len(all_step_recs),
                 _time.strftime('%Y-%m-%dT%H:%M:%S')))
            self._con.executemany(
                'INSERT INTO pnp_euler_steps '
                '(rollout_id, chunk_idx, euler_step, s, u_mean, u_max, a_std_mean) '
                'VALUES (?,?,?,?,?,?,?)',
                [(rollout_id, ci, st['step'], st['s'],
                  st['u_mean'], st['u_max'], st['a_std_mean'])
                 for ci, st in all_step_recs])

    def close(self):
        self._con.close()

    # ── Query helpers ──────────────────────────────────────────────────────

    def query(self, sql, params=()):
        """Return query results as a list of dicts."""
        cur = self._con.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

    def summary(self):
        """Per-task success rate + mean P&P uncertainty (success vs failure)."""
        rows = self.query("""
            SELECT suite, task_idx,
                   COUNT(*)                                              AS n_ep,
                   ROUND(AVG(success)*100, 1)                           AS sr_pct,
                   ROUND(AVG(u_mean_episode), 5)                        AS u_mean_all,
                   ROUND(AVG(CASE WHEN success=0 THEN u_mean_episode END), 5) AS u_mean_fail,
                   ROUND(AVG(CASE WHEN success=1 THEN u_mean_episode END), 5) AS u_mean_succ
            FROM rollouts
            WHERE pnp_enabled=1
            GROUP BY suite, task_idx
            ORDER BY suite, task_idx
        """)
        print(f"{'suite':<18} {'task':>4} {'n':>4} {'sr%':>6} {'u_all':>10} {'u_fail':>10} {'u_succ':>10}")
        print('-' * 70)
        for r in rows:
            print(f"{r['suite']:<18} {r['task_idx']:>4} {r['n_ep']:>4} {r['sr_pct']:>6} "
                  f"{str(r['u_mean_all']):>10} {str(r['u_mean_fail']):>10} {str(r['u_mean_succ']):>10}")
        return rows


# ── Init: open (or create) the DB ────────────────────────────────────────────
import os
DB_PATH = os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else '/content/drive/MyDrive/pi05_results',
                       'rollouts.db')
DB = RolloutDB(DB_PATH)
print(f"Use DB= parameter in run_episode_pnp() to auto-log, or call DB.log_episode() manually.")


RolloutDB: /content/drive/MyDrive/smolvla_results/rollouts.db  (494 existing rollouts)
Use DB= parameter in run_episode_pnp() to auto-log, or call DB.log_episode() manually.


---
## Section 5: P&P Uncertainty & Refinement Evaluation


In [38]:
# ── Configuration ─────────────────────────────────────────────────────
SUITES     = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
N_EPISODES = 10
VERBOSE    = True

MAX_STEPS_MAP = {
    'libero_spatial': 220,
    'libero_object':  280,
    'libero_goal':    300,
    'libero_10':      520,
    'libero_90':      400,
}
print(f'Will evaluate: {SUITES}')
print(f'{N_EPISODES} episodes/task | {N_EPISODES * 10 * len(SUITES)} total episodes')

Will evaluate: ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
10 episodes/task | 400 total episodes


In [39]:
# ── Phase 1: Screening pass — all tasks, P&P disabled, build failure map ─────
# Runs fast (no P&P overhead). Writes to DB. Produces `failed_episodes` list
# consumed by the Phase 2/3 cell below.

import time
from tqdm.notebook import tqdm

PNP_CONFIG.enabled = False
PNP_RECORDER.reset()

failed_episodes = []   # [{suite, task_idx, task_desc, ep_idx, init_state, bddl_path, max_steps}]
all_results_p1  = []
benchmark_dict  = benchmark.get_benchmark_dict()
total_start     = time.time()

for SUITE in tqdm(SUITES, desc='Phase 1 — suites'):
    print(f'\n{"="*60}\nSuite: {SUITE}  [screening]\n{"="*60}')
    task_suite = benchmark_dict[SUITE]()
    max_steps  = MAX_STEPS_MAP.get(SUITE, 300)
    suite_results = []
    suite_start   = time.time()

    for task_idx in tqdm(range(task_suite.n_tasks), desc=f'  {SUITE}', leave=False):
        task        = task_suite.get_task(task_idx)
        init_states = task_suite.get_task_init_states(task_idx)
        bddl_path   = os.path.join(get_libero_path('bddl_files'),
                                   task.problem_folder, task.bddl_file)
        env = OffScreenRenderEnv(
            bddl_file_name=bddl_path, camera_names=CAMERAS,
            camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
            has_offscreen_renderer=True, use_camera_obs=True,
            has_renderer=False, reward_shaping=False,
        )

        n_success = 0
        ep_bar = tqdm(range(min(N_EPISODES, len(init_states))),
                      desc=f'    T{task_idx+1} ({task.language[:35]}...)', leave=False)
        try:
            for ep_idx in ep_bar:
                success, n_steps, elapsed = run_episode_pnp(
                    env, init_states[ep_idx], policy, task.language, max_steps, device,
                    ep_meta={'suite': SUITE, 'task_idx': task_idx, 'episode': ep_idx},
                    db=DB, suite=SUITE, task_idx=task_idx, episode_idx=ep_idx,
                )
                n_success += int(success)
                result = dict(suite=SUITE, task_idx=task_idx,
                              task_desc=task.language, episode=ep_idx,
                              success=success, n_steps=n_steps,
                              elapsed_s=round(elapsed, 2))
                suite_results.append(result)
                all_results_p1.append(result)
                ep_bar.set_postfix(sr=f'{n_success}/{ep_idx+1}',
                                   status='\u2713' if success else '\u2717')
                if not success:
                    failed_episodes.append(dict(
                        suite=SUITE, task_idx=task_idx,
                        task_desc=task.language, ep_idx=ep_idx,
                        init_state=init_states[ep_idx],
                        bddl_path=bddl_path, max_steps=max_steps,
                    ))
        finally:
            env.close()

        sr_str = f'{n_success/min(N_EPISODES, len(init_states)):.0%}'
        tqdm.write(f'  T{task_idx+1} SR: {sr_str} \u2014 {task.language[:55]}...')

    suite_sr  = sum(r['success'] for r in suite_results) / len(suite_results)
    suite_min = (time.time() - suite_start) / 60
    tqdm.write(f'\n{SUITE} SR: {suite_sr:.1%} | {suite_min:.1f} min')

overall_sr  = sum(r['success'] for r in all_results_p1) / len(all_results_p1)
total_min   = (time.time() - total_start) / 60
n_failed    = len(failed_episodes)
print(f'\n{"="*60}')
print(f'OVERALL SR: {overall_sr:.1%}  |  Total time: {total_min:.1f} min')
print(f'Failed episodes: {n_failed} / {len(all_results_p1)}')
print(f'{"="*60}')
print(f'\nRun the Phase 2/3 cell to measure P&P uncertainty on these {n_failed} failures.')


Phase 1 — suites:   0%|          | 0/4 [00:00<?, ?it/s]


Suite: libero_spatial  [screening]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  libero_spatial:   0%|          | 0/10 [00:00<?, ?it/s]

    T1 (pick up the black bowl between the ...):   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

  T8 SR: 90% — put both the alphabet soup and the cream cheese box in ...


    T9 (put both moka pots on the stove...):   0%|          | 0/10 [00:00<?, ?it/s]

  T9 SR: 90% — put both moka pots on the stove...


    T10 (put the yellow and white mug in the...):   0%|          | 0/10 [00:00<?, ?it/s]

  T10 SR: 80% — put the yellow and white mug in the microwave and close...

libero_10 SR: 85.0% | 23.5 min

OVERALL SR: 88.2%  |  Total time: 64.9 min
Failed episodes: 47 / 400

Run the Phase 2/3 cell to measure P&P uncertainty on these 47 failures.


In [ ]:
# ── Reconstruct failed_episodes from DB (use after runtime restart) ───────────
# Phase 1 results are already in the DB; this rebuilds `failed_episodes` by
# re-loading the deterministic LIBERO init_states without re-running the policy.
# Run this cell instead of (or after) Phase 1 when the runtime is fresh.

import sqlite3, json as _json, os

_con = sqlite3.connect(DB_PATH)
_rows = _con.execute(
    "SELECT suite, task_idx, task_desc, episode_idx "
    "FROM rollouts WHERE pnp_enabled=0 AND success=0 "
    "ORDER BY suite, task_idx, episode_idx"
).fetchall()
_con.close()

if not _rows:
    print("No Phase 1 failures found in DB — run Phase 1 first.")
else:
    benchmark_dict = benchmark.get_benchmark_dict()
    failed_episodes = []

    # Group by (suite, task_idx) to load init_states once per task
    from itertools import groupby
    for (suite, task_idx), grp in groupby(_rows, key=lambda r: (r[0], r[1])):
        episodes = list(grp)
        task_suite  = benchmark_dict[suite]()
        task        = task_suite.get_task(task_idx)
        init_states = task_suite.get_task_init_states(task_idx)
        bddl_path   = os.path.join(get_libero_path('bddl_files'),
                                   task.problem_folder, task.bddl_file)
        max_steps   = MAX_STEPS_MAP.get(suite, 300)

        for _, _, task_desc, ep_idx in episodes:
            failed_episodes.append(dict(
                suite=suite, task_idx=task_idx,
                task_desc=task_desc, ep_idx=ep_idx,
                init_state=init_states[ep_idx],
                bddl_path=bddl_path, max_steps=max_steps,
            ))

    print(f"Reconstructed {len(failed_episodes)} failed episodes from DB.")
    print("Ready to run the Phase 2/3 cell.")


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7

In [40]:
# ── Phases 2 & 3: P&P on failed episodes only ────────────────────────────────
# Reads `failed_episodes` built by Phase 1. Groups by task to reuse envs.
# Both phases write to DB (separate rollout_ids via pnp_config hash), so
# re-running with a different STEP_CONFIGS entry is safe and non-destructive.
#
#   Phase 2 — mode="uncertainty": non-invasive, measures U without altering trajectory
#   Phase 3 — mode="both":        refines trajectory, tests if P&P recovers failures
#
# With num_inference_steps=10 the noise schedule is:
#   step 0 → s=1.0 (pure noise)  — refinement is a no-op here (a_hat dropped); OK for uncertainty
#   step 1 → s=0.9               ← early/high-noise range
#   step 2 → s=0.8
#   step 3 → s=0.7
#   step 4 → s=0.6               ← mid-denoising range
#   step 5 → s=0.5
#   step 6 → s=0.4
#   step 7 → s=0.3               ← late/near-clean range
#   step 8 → s=0.2
#   step 9 → s=0.1

import time
from itertools import groupby
from tqdm.notebook import tqdm

RUN_PHASE_2 = True    # uncertainty measurement (non-invasive)
RUN_PHASE_3 = True    # trajectory refinement

# ── Step-index configurations to sweep ───────────────────────────────────────
# Each entry is run as a separate experiment; the DB separates them via rollout_id hashing.
# Uncomment additional entries to sweep multiple step positions in one run.
STEP_CONFIGS = [
    # (1, 2),   # early  — high noise, maximum malleability (original)
    (4, 5), # mid    — more reliable a_hat prediction
    (7, 8), # late   — near-clean, precise but small perturbation
]
PNP_K = 3

if not failed_episodes:
    print('No failed episodes from Phase 1 — nothing to run.')
else:
    phases = []
    if RUN_PHASE_2:
        phases.append((2, 'uncertainty'))
    if RUN_PHASE_3:
        phases.append((3, 'both'))

    for step_indices in STEP_CONFIGS:
        steps_label = str(list(step_indices))
        s_vals = [round(1.0 - i / 10, 1) for i in step_indices]
        print(f'\n{"#"*60}')
        print(f'Step config: steps={steps_label}  (s={s_vals})')
        print(f'{"#"*60}')

        for phase_num, pnp_mode in phases:
            print(f'\n{"="*60}')
            print(f'Phase {phase_num}: pnp mode="{pnp_mode}" | steps={steps_label} | {len(failed_episodes)} episodes')
            print(f'{"="*60}')

            PNP_CONFIG.enabled        = True
            PNP_CONFIG.mode           = pnp_mode
            PNP_CONFIG.step_indices   = step_indices
            PNP_CONFIG.time_min       = None
            PNP_CONFIG.num_iterations = PNP_K
            PNP_RECORDER.reset()

            # Group by (suite, task_idx) so each env is created once per task
            sorted_fails = sorted(failed_episodes, key=lambda x: (x['suite'], x['task_idx']))
            phase_results = []
            phase_start = time.time()

            for (suite, task_idx), group_iter in groupby(
                    sorted_fails, key=lambda x: (x['suite'], x['task_idx'])):
                episodes = list(group_iter)
                env = OffScreenRenderEnv(
                    bddl_file_name=episodes[0]['bddl_path'], camera_names=CAMERAS,
                    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
                    has_offscreen_renderer=True, use_camera_obs=True,
                    has_renderer=False, reward_shaping=False,
                )
                ep_bar = tqdm(episodes,
                              desc=f'  P{phase_num} steps={steps_label} T{task_idx+1} ({episodes[0]["task_desc"][:30]}...)',
                              leave=False)
                try:
                    for ep_info in ep_bar:
                        success, n_steps, elapsed = run_episode_pnp(
                            env, ep_info['init_state'], policy,
                            ep_info['task_desc'], ep_info['max_steps'], device,
                            ep_meta={'suite': suite, 'task_idx': task_idx,
                                     'episode': ep_info['ep_idx'], 'phase': phase_num,
                                     'step_indices': list(step_indices)},
                            db=DB, suite=suite, task_idx=task_idx,
                            episode_idx=ep_info['ep_idx'],
                        )
                        phase_results.append(dict(
                            suite=suite, task_idx=task_idx,
                            ep_idx=ep_info['ep_idx'], success=success,
                            n_steps=n_steps,
                        ))
                        ep_bar.set_postfix(status='\u2713' if success else '\u2717')
                finally:
                    env.close()

            phase_sr  = sum(r['success'] for r in phase_results) / len(phase_results)
            phase_min = (time.time() - phase_start) / 60
            print(f'Phase {phase_num} steps={steps_label} SR on failures: {phase_sr:.1%}  |  {phase_min:.1f} min')

    PNP_CONFIG.enabled = False
    print('\nDone. DB summary:')
    DB.summary()


############################################################
Step config: steps=[4, 5]  (s=[0.6, 0.5])
############################################################

Phase 2: pnp mode="uncertainty" | steps=[4, 5] | 47 episodes


  P2 steps=[4, 5] T1 (put both the alphabet soup and...):   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  P2 steps=[4, 5] T3 (turn on the stove and put the ...):   0%|          | 0/1 [00:00<?, ?it/s]

  P2 steps=[4, 5] T4 (put the black bowl in the bott...):   0%|          | 0/2 [00:00<?, ?it/s]

...

  P3 steps=[7, 8] T4 (pick up the black bowl on the ...):   0%|          | 0/1 [00:00<?, ?it/s]

  P3 steps=[7, 8] T5 (pick up the black bowl in the ...):   0%|          | 0/2 [00:00<?, ?it/s]

  P3 steps=[7, 8] T6 (pick up the black bowl on the ...):   0%|          | 0/6 [00:00<?, ?it/s]

  P3 steps=[7, 8] T9 (pick up the black bowl next to...):   0%|          | 0/3 [00:00<?, ?it/s]

Phase 3 steps=[7, 8] SR on failures: 59.6%  |  10.6 min

Done. DB summary:
suite              task    n    sr%      u_all     u_fail     u_succ
----------------------------------------------------------------------
libero_10             0   30   66.7    0.02061    0.02306    0.01938
libero_10             2    6  100.0    0.01844       None    0.01844
libero_10             3   12   83.3    0.02094    0.02201    0.02073
libero_10             4   12   41.7    0.01423    0.01386    0.01474
libero_10             5    6   50.0    0.03031     0.0373    0.02332
libero_10             7    6    0.0    0.02816    0.02816       None
libero_10             8    6   33.3    0.01604    0.01899    0.01015
libero_10             9   12   91.7    0.02943    0.03196     0.0292
libero_goal           1    6  100.0    0.01836       None    0.01836
libero_goal           2   12   83.3    0.02827    0.03714     0.0265
libero_goal           3   12   58.3    0.02582    0.03403    0.01996
libero_goal           5   

In [1]:
!ls

drive  rollout.mp4  sample_data


In [41]:
from google.colab import runtime
runtime.unassign()